# News Topic Classifier from First Principles in PyTorch

Welcome to the complete, self-contained notebook to train and evaluate a **News Topic Classifier using BERT from first principles in PyTorch**.

### Instructions to run on Kaggle:
1. In the right panel of your Kaggle notebook, change **Accelerator** to **GPU T4 x2** or **GPU T4**.
2. Run the cells sequentially to install dependencies, construct classes, run the training pipeline, and inspect scientific diagnostics.

In [ ]:
# Step 1: Install required Hugging Face & scientific dependencies
!pip install -q transformers datasets scikit-learn accelerate

In [ ]:
# Step 2: System Imports & Hardware Diagnostics
import torch
import torch.nn as nn
import numpy as np
import os
import re
import time
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device.type.upper()}")
if device.type == "cuda":
    print(f"GPU Details: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 3: First Principles Preprocessing Class (Dataset, HTML Cleaning, Dynamic Padding Collator)
MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 128

def clean_text(text: str) -> str:
    # Remove HTML tags using regex
    clean = re.sub(r'<[^>]+>', ' ', text)
    # Replace multiple spaces with single space
    clean = re.sub(r'\s+', ' ', clean)
    return clean.strip()

class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer_name=MODEL_NAME, max_length=MAX_LENGTH):
        self.texts = [clean_text(text) for text in texts]
        self.labels = labels
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        # Tokenize without static padding (for Dynamic Collation)
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_attention_mask=True
        )
        return {
            "input_ids": torch.tensor(encoding["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(encoding["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(label, dtype=torch.long)
        }

class DynamicPaddingCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, batch):
        input_ids_list = [item["input_ids"] for item in batch]
        attention_mask_list = [item["attention_mask"] for item in batch]
        labels = torch.stack([item["labels"] for item in batch])

        # Dynamic batch-level padding
        padded_input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids_list,
            batch_first=True,
            padding_value=self.tokenizer.pad_token_id
        )
        padded_attention_mask = torch.nn.utils.rnn.pad_sequence(
            attention_mask_list,
            batch_first=True,
            padding_value=0
        )
        return {
            "input_ids": padded_input_ids,
            "attention_mask": padded_attention_mask,
            "labels": labels
        }

In [ ]:
# Step 4: Custom NewsClassifier Model inheriting from nn.Module
# Implements the last 4 hidden states pooling concatenation
class NewsClassifier(nn.Module):
    def __init__(self, model_name="bert-base-uncased", num_classes=4, freeze_backbone=False):
        super(NewsClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.hidden_dim = self.bert.config.hidden_size
        
        # Inputs: 768 hidden features * 4 concatenated layers = 3072 features
        self.classification_head = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(self.hidden_dim * 4, self.hidden_dim),
            nn.LayerNorm(self.hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(self.hidden_dim, num_classes)
        )
        
        if freeze_backbone:
            for param in self.bert.parameters():
                param.requires_grad = False
                
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        
        hidden_states = outputs.hidden_states
        cls_9 = hidden_states[-4][:, 0, :]
        cls_10 = hidden_states[-3][:, 0, :]
        cls_11 = hidden_states[-2][:, 0, :]
        cls_12 = hidden_states[-1][:, 0, :]
        
        pooled_output = torch.cat((cls_12, cls_11, cls_10, cls_9), dim=-1)
        return self.classification_head(pooled_output)

In [ ]:
# Step 5: Custom PyTorch Training Epoch & Validation Loop
def train_one_epoch(model, dataloader, optimizer, scheduler, loss_fn, scaler, device, grad_accum_steps=2):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    
    for step, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        # Autocast AMP
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            logits = model(input_ids, attention_mask)
            loss = loss_fn(logits, labels)
            loss = loss / grad_accum_steps
            
        if device.type == "cuda":
            scaler.scale(loss).backward()
        else:
            loss.backward()
            
        total_loss += loss.item() * grad_accum_steps
        
        # Gradient Accumulation & Clipping
        if (step + 1) % grad_accum_steps == 0 or (step + 1) == len(dataloader):
            if device.type == "cuda":
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                
            scheduler.step()
            optimizer.zero_grad()
            
    return total_loss / len(dataloader)

def validate(model, dataloader, loss_fn, device):
    model.eval()
    total_loss = 0
    correct_predictions = 0
    total_samples = 0
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            logits = model(input_ids, attention_mask)
            loss = loss_fn(logits, labels)
            total_loss += loss.item()
            
            preds = logits.argmax(dim=-1)
            correct_predictions += (preds == labels).sum().item()
            total_samples += labels.size(0)
            
    return total_loss / len(dataloader), correct_predictions / total_samples

In [ ]:
# Step 6: Dataset Downloading, DataLoader Building, and Model Training Loop
print("Loading dataset from Hugging Face...")
ag_news = load_dataset("ag_news")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = DynamicPaddingCollator(tokenizer)

train_set = NewsDataset(ag_news["train"]["text"][:20000], ag_news["train"]["label"][:20000]) # Slicing for fast Kaggle demonstration
test_set = NewsDataset(ag_news["test"]["text"][:2000], ag_news["test"]["label"][:2000])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True, collate_fn=collator, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False, collate_fn=collator, pin_memory=True)

print("Initializing Custom Model...")
model = NewsClassifier(num_classes=4, freeze_backbone=False).to(device)

# Optimizer, Schedulers, and Scalers
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1) # Label smoothing regularization
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

EPOCHS = 3
num_training_steps = (len(train_loader) // 2) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(num_training_steps * 0.1), num_training_steps=num_training_steps)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

print("--- Starting Training Loop ---")
best_acc = 0.0
for epoch in range(EPOCHS):
    epoch_start = time.time()
    t_loss = train_one_epoch(model, train_loader, optimizer, scheduler, loss_fn, scaler, device)
    v_loss, v_acc = validate(model, test_loader, loss_fn, device)
    print(f"Epoch {epoch+1} | Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f} | Val Acc: {v_acc:.2%} | Time: {time.time()-epoch_start:.1f}s")
    
    if v_acc > best_acc:
        best_acc = v_acc
        print(f" --> Saving new best checkpoint...")
        torch.save(model.state_dict(), "kaggle_news_classifier.bin")

In [ ]:
# Step 7: Scientific Metrics Reporting & High-Confidence Error Diagnostics
LABELS = ["World", "Sports", "Business", "Sci/Tech"]

model.load_state_dict(torch.load("kaggle_news_classifier.bin"))
model.eval()

all_preds = []
all_labels = []
all_probs = []
raw_texts = ag_news["test"]["text"][:2000]

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        
        logits = model(input_ids, attention_mask)
        probs = torch.softmax(logits, dim=-1)
        preds = logits.argmax(dim=-1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch["labels"].cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

print(classification_report(all_labels, all_preds, target_names=LABELS, digits=4))

# Diagnostics: Confidence Mistakes
incorrect_idx = np.where(all_preds != all_labels)[0]
incorrect_probs = [all_probs[idx][all_preds[idx]] for idx in incorrect_idx]
sorted_errs = np.argsort(incorrect_probs)[::-1]

print("\n--- TOP 5 HIGH-CONFIDENCE ERRORS ---")
for i in range(min(5, len(sorted_errs))):
    pos = incorrect_idx[sorted_errs[i]]
    print(f"{i+1}. Confidence: {all_probs[pos][all_preds[pos]]:.2%}")
    print(f"   True Label     : {LABELS[all_labels[pos]]}")
    print(f"   Predicted Label: {LABELS[all_preds[pos]]}")
    print(f"   Headline       : \"{raw_texts[pos]}\"\n")

### Step 8: Dynamic INT8 CPU Quantization Benchmarking

Quantization will convert model parameters in linear modules from float32 to int8. Run the code below to see the impact of this optimization on inference speed.

In [ ]:
model.to("cpu")
model.eval()

# Apply Post-Training Dynamic Quantization
quantized_model = torch.quantization.quantize_dynamic(
    model, 
    {nn.Linear}, 
    dtype=torch.qint8
)

# Benchmark Latency (Dynamic Quantization vs. Original Model)
sample_input = torch.randint(0, 1000, (1, 128))
sample_mask = torch.ones((1, 128), dtype=torch.long)

# Standard Model on CPU
start = time.time()
for _ in range(50):
    with torch.no_grad():
        _ = model(sample_input, sample_mask)
cpu_latency = (time.time() - start) * 1000 / 50

# Quantized Model on CPU
start = time.time()
for _ in range(50):
    with torch.no_grad():
        _ = quantized_model(sample_input, sample_mask)
quant_latency = (time.time() - start) * 1000 / 50

print(f"Average CPU Latency (Standard Model) : {cpu_latency:.2f} ms")
print(f"Average CPU Latency (Quantized Model): {quant_latency:.2f} ms")
print(f"Inference speedup: {cpu_latency / quant_latency:.2f}x!")

### Step 9: Export and Upload Trained Model to Hugging Face Hub

We retrieve your Hugging Face WRITE access token securely from Kaggle Secrets (Label: `HF_TOKEN`) to avoid hardcoding credentials in your notebook, programmatically create the model repository, and upload your trained model weights (`kaggle_news_classifier.bin`).

**Instructions for Kaggle Secrets:**
1. In the top menu of your Kaggle notebook, click **Add-ons** -> **Secrets**.
2. Add a new secret with Label: `HF_TOKEN` and Value: *[Your Hugging Face WRITE access token]*.
3. Turn on the checkmark next to `HF_TOKEN` to share it with this notebook.

In [ ]:
# Install huggingface_hub
!pip install -q huggingface_hub

from huggingface_hub import HfApi, create_repo, login
import os

# 1. Fetch token from Kaggle Secrets securely (avoids hardcoding in code)
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
except Exception:
    # Fallback to local environment variable if running outside Kaggle
    HF_TOKEN = os.getenv("HF_TOKEN")

REPO_NAME = "news-topic-classifier-bert"

if not HF_TOKEN:
    print("❌ ERROR: 'HF_TOKEN' not found in Kaggle Secrets! Please add a secret with label 'HF_TOKEN' containing your Hugging Face WRITE access token.")
else:
    # Log in to remote server
    login(token=HF_TOKEN)
    
    api = HfApi()
    username = api.whoami()["name"]
    full_repo_id = f"{username}/{REPO_NAME}"
    
    # Create repo on the hub
    create_repo(repo_id=full_repo_id, repo_type="model", exist_ok=True)
    print(f"✔ Repository {full_repo_id} verified/created on the Hub.")
    
    # Upload weights file
    local_weights = "kaggle_news_classifier.bin"
    if os.path.exists(local_weights):
        print(f"Uploading weights from local file {local_weights} to remote {full_repo_id}...")
        api.upload_file(
            path_or_fileobj=local_weights,
            path_in_repo="pytorch_model.bin",
            repo_id=full_repo_id,
            repo_type="model"
        )
        print(f"\n🎉 SUCCESS! Your model is live at: https://huggingface.co/{full_repo_id}")
    else:
        print(f"❌ ERROR: Weights file '{local_weights}' not found. Make sure you run Step 6 successfully first.")